# Train a Spiking Neural Network on MNIST in 5 Minutes

**Nuro SDK** — The universal SDK for spiking neural networks.

In this notebook, you'll:
1. Build an SNN with LIF neurons
2. Train it on MNIST using surrogate gradients
3. Achieve >90% accuracy in under 5 minutes

No neuromorphic hardware needed — everything runs on a standard GPU.

[![GitHub](https://img.shields.io/badge/GitHub-Vantar--AI%2Fnuro-black)](https://github.com/Vantar-AI/nuro)
[![License](https://img.shields.io/badge/License-Apache%202.0-blue)](https://github.com/Vantar-AI/nuro/blob/main/LICENSE)

## 1. Install Nuro

In [ ]:
!pip install -q nuro[gpu] torchvision

## 2. Load MNIST

Standard MNIST — 28x28 grayscale digits. We'll flatten to 784 and encode pixel intensities as spike rates.

In [ ]:
import torch
import torch.nn.functional as F
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_dataset = datasets.MNIST("./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST("./data", train=False, transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=256)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

## 3. Define the Spiking Network

Three populations of LIF neurons:
- **Input** (784) — one neuron per pixel
- **Hidden** (256) — hidden layer with 20ms time constant
- **Output** (10) — one neuron per digit class

This is the entire network definition. It never changes regardless of which hardware you deploy to.

In [ ]:
import nuro

print(f"Nuro version: {nuro.__version__}")

# Define populations (neuron groups)
inp_pop = nuro.Population(size=784, dynamics="lif", params={"tau": 10e-3})
hidden  = nuro.Population(size=256, dynamics="lif", params={"tau": 20e-3})
output  = nuro.Population(size=10,  dynamics="lif", params={"tau": 5e-3})

# Define connections
c1 = nuro.Connection(source=inp_pop, target=hidden, pattern="dense")
c2 = nuro.Connection(source=hidden,  target=output, pattern="dense")

print("Network: 784 -> 256 -> 10 LIF neurons")
print("Connections: 2 dense layers")

## 4. Train with Surrogate Gradients

Spiking neurons are non-differentiable (binary spike/no-spike). Nuro uses **surrogate gradients** — smooth approximations of the spike function during the backward pass. This lets you train SNNs with standard backpropagation-through-time (BPTT).

The training loop looks exactly like PyTorch — because it is PyTorch under the hood.

In [ ]:
EPOCHS = 5
DURATION = 0.05  # 50ms simulation per sample
DT = 1e-3        # 1ms timesteps
LR = 2e-3

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Training on: {device}")

for epoch in range(EPOCHS):
    correct = 0
    total = 0
    epoch_loss = 0.0

    for batch_idx, (data, target) in enumerate(train_loader):
        data = data.view(-1, 784).to(device)
        target = target.to(device)
        batch_size = data.shape[0]

        # Encode pixels as spike rates (repeat across timesteps)
        num_steps = int(DURATION / DT)
        spike_input = (torch.rand(num_steps, batch_size, 784, device=device) < data.unsqueeze(0).abs()).float()

        # Build graph with this batch's input
        inp = nuro.Input(population=inp_pop, data=spike_input)
        graph = nuro.Graph([inp_pop, hidden, output], [c1, c2], inputs=[inp])

        # Compile with surrogate gradients
        model = nuro.compile(graph, target="gpu", requires_grad=True, surrogate="atan")

        # Reuse weights from previous batch (except first)
        if batch_idx > 0 or epoch > 0:
            model.snn.load_state_dict(state_dict)

        optimizer = torch.optim.Adam(model.snn.parameters(), lr=LR)
        optimizer.zero_grad()

        # Forward pass — simulate the SNN
        out = model.run(duration=DURATION, dt=DT, batch_size=batch_size)

        # Spike counts as logits
        logits = out[output.id]
        loss = F.cross_entropy(logits, target)

        # Backward pass
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.snn.parameters(), 1.0)
        optimizer.step()

        state_dict = model.snn.state_dict()

        # Track accuracy
        pred = logits.argmax(dim=1)
        correct += (pred == target).sum().item()
        total += batch_size
        epoch_loss += loss.item()

        if batch_idx % 100 == 0:
            print(f"  Epoch {epoch+1}/{EPOCHS} | Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}")

    acc = 100.0 * correct / total
    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} — Accuracy: {acc:.1f}% | Avg Loss: {avg_loss:.4f}")

print(f"\nTraining complete! Final accuracy: {acc:.1f}%")

## 5. Evaluate on Test Set

In [ ]:
correct = 0
total = 0

with torch.no_grad():
    for data, target in test_loader:
        data = data.view(-1, 784).to(device)
        target = target.to(device)
        batch_size = data.shape[0]

        num_steps = int(DURATION / DT)
        spike_input = (torch.rand(num_steps, batch_size, 784, device=device) < data.unsqueeze(0).abs()).float()

        inp = nuro.Input(population=inp_pop, data=spike_input)
        graph = nuro.Graph([inp_pop, hidden, output], [c1, c2], inputs=[inp])
        model = nuro.compile(graph, target="gpu", requires_grad=False)
        model.snn.load_state_dict(state_dict)

        out = model.run(duration=DURATION, dt=DT, batch_size=batch_size)
        pred = out[output.id].argmax(dim=1)
        correct += (pred == target).sum().item()
        total += batch_size

test_acc = 100.0 * correct / total
print(f"Test Accuracy: {test_acc:.1f}%")

## 6. Deploy to Neuromorphic Hardware

The same network — trained above — can deploy to real neuromorphic chips with **one line change**.

```python
# Save GPU weights
model.save("mnist_snn.pt")

# Deploy to Intel Loihi 2
loihi_model = nuro.compile(graph, target="loihi", weights_from="mnist_snn.pt")
loihi_model.run(duration=0.05)

# Deploy to SpiNNaker 2
s2_model = nuro.compile(graph, target="spinnaker2", weights_from="mnist_snn.pt")

# Deploy to BrainChip Akida
akida_model = nuro.compile(graph, target="akida", weights_from="mnist_snn.pt")
```

Weights are automatically quantized for each hardware target. No manual conversion needed.

## What's Next?

- [02 — Convert a PyTorch CNN to SNN](./02_convert_pytorch_to_snn.ipynb)
- [03 — Deploy to Neuromorphic Hardware](./03_deploy_to_hardware.ipynb)
- [GitHub](https://github.com/Vantar-AI/nuro) | [Website](https://vantar.xyz) | [Docs](https://github.com/Vantar-AI/nuro/tree/main/docs)